## Image Rotation

In [ ]:
# import os
# import cv2
# from concurrent.futures import ThreadPoolExecutor
# from tqdm import tqdm

# def rotate_image(img_path):
#     """Rotate a single image 90 degrees counter-clockwise"""
#     img = cv2.imread(img_path)
#     if img is None:
#         return f"Failed: {img_path}"
    
#     # OpenCV rotate 90 degrees counter-clockwise
#     rotated = cv2.rotate(img, cv2.ROTATE_90_COUNTERCLOCKWISE)
    
#     # Get filename and create output path
#     filename = os.path.basename(img_path)
#     output_path = os.path.join("img/segments_rotated", filename)
    
#     cv2.imwrite(output_path, rotated)
#     return f"Rotated: {filename}"

# # Get list of image files from input directory
# input_dir = "img/segments"
# image_extensions = ('.png', '.jpg', '.jpeg', '.gif', '.bmp')

# image_files = [
#     os.path.join(input_dir, f) 
#     for f in os.listdir(input_dir) 
#     if f.lower().endswith(image_extensions)
# ]

# # Create output directory if it doesn't exist
# os.makedirs("img/segments_rotated", exist_ok=True)

# # Process images in parallel using ThreadPoolExecutor (works in Jupyter)
# with ThreadPoolExecutor(max_workers=4) as executor:
#     results = tqdm(executor.map(rotate_image, image_files), total=len(image_files), desc="Rotating images")

# for result in results:
#     print(result)

# print(f"All {len(image_files)} images rotated successfully!")

## Review File Generator

In [3]:
import pandas as pd

df = pd.read_csv('spine_extraction_results_incremental.csv')
df.file = "http://localhost:8000/" + df.file
df.to_csv('spine_extraction_results_incremental_full_url.csv', index=False)
df

,file,title,author,call_no
0,http://localhost:8000/03d4dc70-IMG_2817_spine_...,NEWLY INDUSTRIALIZING COUNTRIES AND THE POLITI...,ED BY JERGER CARLESON AND TIMOTHY M. SHAW,HF 1413 .N497 1988
1,http://localhost:8000/03d4dc70-IMG_2817_spine_...,Fair Trade For All,Stiglitz and Charlton,HF 1413 $85 2005
2,http://localhost:8000/03d4dc70-IMG_2817_spine_...,Third World Strategy,Gauhar,HF 1413 T5 1983
3,http://localhost:8000/03d4dc70-IMG_2817_spine_...,U.S. FOREIGN POLICY AND THE THIRD WORLD: AGEND...,Lewis Kallab,HF 1413 U5x 1983
4,http://localhost:8000/03d4dc70-IMG_2817_spine_...,U.S. FOREIGN POLICY and the THIRD WORLD: AGEND...,NaN,HF 1413 U535 1985
...,...,...,...,...
2727,http://localhost:8000/e582243e-IMG_3614_spine_...,MEET THE PRESS,KRAUS,E 743 M4
2728,http://localhost:8000/e582243e-IMG_3614_spine_...,MEET THE PRESS,12,E 743 M4
2729,http://localhost:8000/e582243e-IMG_3614_spine_...,MEET THE PRESS,13,E 743 M4
2730,http://localhost:8000/e582243e-IMG_3614_spine_...,MEET THE PRESS,KRAUS,E 743 H4


## Training File Generator

In [7]:
import pandas as pd
train_df = pd.read_csv('input/llm_spine_parse_train_csv.csv')
train_df.head(10)

,title,file,call_no,author
0,NEWLY INDUSTRIALIZING COUNTRIES AND T~ POLITIC...,03d4dc70-IMG_2817_spine_000.png,HF 1413 .N497 1988,NaN
1,Fair Trade For All,03d4dc70-IMG_2817_spine_001.png,HF 1413 S85 2005,Stiglitz and Charlton
2,Third World Strategy,03d4dc70-IMG_2817_spine_002.png,HF 1413 T5 1983,Gauhar
3,U.S. FOREIGN POLICY AND THE THIRD WORLD: AGEND...,03d4dc70-IMG_2817_spine_003.png,HF 1413 U5x 1983,Lewis Kallab
4,U.S. FOREIGN POLICY and the THIRD WORLD: AGEND...,03d4dc70-IMG_2817_spine_004.png,HF 1413 U535 1985,NaN
5,Neocolonialism; Methods and Manoeuvr~,03d4dc70-IMG_2817_spine_005.png,HF 1413 V3213,V.VAKHRUSHEV
6,The Developing Economies and the International...,03d4dc70-IMG_2817_spine_006.png,HF 1413 V45,S. Venu
7,VIEWS FROM THE SOUTH,03d4dc70-IMG_2817_spine_007.png,HF 1413 .V53 2000,NaN
8,NaN,03d4dc70-IMG_2817_spine_008.png,HF 1413 W4,NaN
9,World Trade Competition,03d4dc70-IMG_2817_spine_009.png,HF 1413 W67,Center for Strategic and International Studies


In [8]:
import json
import base64
import os
from pathlib import Path

# Take first 100 rows of train_df for training data preparation
sample_df = train_df

# Create training data in the format for multimodal LLM fine-tuning
# Each row will include base64-encoded image data

training_data = []

def image_to_base64(image_path):
    """Convert image file to base64 string"""
    with open(image_path, "rb") as image_file:
        return base64.b64encode(image_file.read()).decode('utf-8')

for idx, row in sample_df.iterrows():
    # Extract filename from the 'file' column
    filename = Path(row['file']).name
    image_path = f"img/segments_512/{filename}"
    
    # Check if image exists
    if not os.path.exists(image_path):
        print(f"Warning: {image_path} not found, skipping...")
        continue
    
    # Convert image to base64
    image_base64 = image_to_base64(image_path)
    
    # Create the expected JSON output
    expected_json = {
        "title": row['title'] if pd.notna(row['title']) else None,
        "author": row['author'] if pd.notna(row['author']) else None,
        "call_no": row['call_no'] if pd.notna(row['call_no']) else None
    }
    
    # Create a training example with base64-encoded image (image last for readability)
    training_example = {
        "prompt": "Extract the book information from this spine image and return it as JSON with keys: title, author, call_no.",
        "completion": json.dumps(expected_json),
        "image": f"data:image/jpeg;base64,{image_base64}"
    }
    
    training_data.append(training_example)

# Convert to DataFrame for easy viewing (excluding large base64 strings for readability)
training_data_preview = [{k: v if k != 'image' else f"<base64 image data: {len(v)} chars>" 
                          for k, v in item.items()} for item in training_data]
training_data_df = pd.DataFrame(training_data_preview)

# Save to JSONL format (common format for LLM training)
with open('llm_training_data_multimodal.jsonl', 'w') as f:
    for item in training_data:
        f.write(json.dumps(item) + '\n')

print(f"Created {len(training_data)} training examples")
print("\nFirst 3 examples (image data truncated for display):")
print(training_data_df.head(3))
print(f"\nTraining data saved to 'llm_training_data_multimodal.jsonl'")

Created 2730 training examples

First 3 examples (image data truncated for display):
                                              prompt  \
0  Extract the book information from this spine i...   
1  Extract the book information from this spine i...   
2  Extract the book information from this spine i...   

                                          completion  \
0  {"title": "NEWLY INDUSTRIALIZING COUNTRIES AND...   
1  {"title": "Fair Trade For All", "author": "Sti...   
2  {"title": "Third World Strategy", "author": "G...   

                              image  
0  <base64 image data: 56299 chars>  
1  <base64 image data: 46995 chars>  
2  <base64 image data: 52227 chars>  

Training data saved to 'llm_training_data_multimodal.jsonl'


## Training (CUDA LoRA)

In [6]:
# CUDA LoRA training will run on a GPU box, not here.
print("CUDA LoRA training is configured to run on a CUDA machine.")
print("Use the script: python train_qwen_vl_lora_cuda.py --help")


CUDA LoRA training is configured to run on a CUDA machine.
Use the script: python train_qwen_vl_lora_cuda.py --help


## CUDA LoRA Training (Run on GPU)
This project now targets CUDA-based LoRA fine-tuning for Qwen2.5-VL. Use `train_qwen_vl_lora_cuda.py` on a GPU box.

Steps (on a CUDA machine):

1. Create and activate a virtualenv.
2. Install requirements, then install PyTorch with the correct CUDA wheel.
3. Run the trainer with a tiny subset to validate.


In [ ]:
# Example commands to run on a CUDA machine (not executed here)
print("Suggested setup on CUDA box:")
print("\n1) python -m venv .venv && source .venv/bin/activate")
print("2) pip install -r requirements.txt")
print("3) Install PyTorch with CUDA from https://pytorch.org/get-started/locally/")
print("4) Run training (tiny quick run):")
print("   python train_qwen_vl_lora_cuda.py --model_id Qwen/Qwen2.5-VL-7B-Instruct-AWQ \\")
print("      --data_file llm_training_data_multimodal.jsonl --output_dir checkpoints/qwen2_5_vl_adq_lora \\")
print("      --max_samples 20 --epochs 1 --batch_size 1 --lr 1e-5")


## Dev Run (50 Samples)
Run a quick validation on 50 samples with batch_size=1 and gradient accumulation to ensure stable collation and training.

In [4]:
# 50-sample dev run (stable settings)
print("Starting 50-sample dev run with batch_size=1 and GA=4...")
! python train_qwen_vl_lora_qlora.py \
  --model_id "Qwen/Qwen2.5-VL-7B-Instruct" \
  --data_file llm_training_data_multimodal.jsonl \
  --max_samples 50 \
  --output_dir checkpoints/qwen2_5_vl_lora_dev \
  --num_epochs 1 \
  --batch_size 1 \
  --gradient_accumulation_steps 4 \
  --lr 2e-4 \
  --warmup_steps 50 \
  --save_steps 50 \
  --logging_steps 10

Starting 50-sample dev run with batch_size=1 and GA=4...

QWEN2.5-VL QLORA FINE-TUNING PIPELINE


QWEN2.5-VL QLORA FINE-TUNING PIPELINE

Loaded 50 training examples
[1/3] Setting up QLoRA configuration...
Loaded 50 training examples
[1/3] Setting up QLoRA configuration...
The image processor of type `Qwen2VLImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. Note that this behavior will be extended to all models in a future release.
The image processor of type `Qwen2VLImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. Note that this behavior

In [ ]:
# Step 1: Fine-tune with QLoRA (stable settings: batch_size=1, GA=4)
print("="*60)
print("STEP 1: QLoRA FINE-TUNING (FULL DATA)")
print("="*60)
! python train_qwen_vl_lora_qlora.py \
    --model_id "Qwen/Qwen2.5-VL-7B-Instruct" \
    --data_file llm_training_data_multimodal.jsonl \
    --output_dir checkpoints/qwen2_5_vl_lora_full \
    --num_epochs 3 \
    --batch_size 1 \
    --gradient_accumulation_steps 4 \
    --lr 2e-4 \
    --warmup_steps 200 \
    --save_steps 200 \
    --logging_steps 20

TRAINING CONFIGURATION SUMMARY
Data file: llm_training_data_multimodal.jsonl
Number of examples: 20
Output directory: checkpoints/qwen2_5_vl_awq_lora
Epochs: 1
Batch size: 1
Learning rate: 1e-05
The image processor of type `Qwen2VLImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. Note that this behavior will be extended to all models in a future release.
/home/quist/source/llm_train/venv/lib/python3.12/site-packages/transformers/models/auto/modeling_auto.py:2284: FutureWarning: The class `AutoModelForVision2Seq` is deprecated and will be removed in v5.0. Please use `AutoModelForImageTextToText` instead.
  warnings.warn(
`torch.bfloat16` is not supported for AWQ CUDA/XPU kernels yet. Casting to `torch.float16`.
/home/quist/source/llm_train/venv/lib/python3.12/sit

In [ ]:
# Step 2: Merge LoRA adapters into base model
# Note: Using merge_lora_simple.py (manual merger) to avoid PEFT hang issues
print("\n" + "="*60)
print("STEP 2: MERGING LORA ADAPTERS")
print("="*60)
! python merge_lora_simple.py \
    --base_model_id "Qwen/Qwen2.5-VL-7B-Instruct" \
    --adapter_dir checkpoints/qwen2_5_vl_lora_full/adapter_model \
    --output_dir checkpoints/qwen2_5_vl_lora_full/merged

In [ ]:
# Step 2 (DEV): Merge LoRA adapters from dev run
# Note: Using merge_lora_simple.py (manual merger) to avoid PEFT hang issues
print("\n" + "="*60)
print("STEP 2 (DEV): MERGING LORA ADAPTERS")
print("="*60)
! python merge_lora_simple.py \
    --base_model_id "Qwen/Qwen2.5-VL-7B-Instruct" \
    --adapter_dir checkpoints/qwen2_5_vl_lora_dev/adapter_model \
    --output_dir checkpoints/qwen2_5_vl_lora_dev/merged


STEP 2 (DEV): MERGING LORA ADAPTERS

MERGING LORA ADAPTERS INTO BASE MODEL

[1/3] Loading base model: Qwen/Qwen2.5-VL-7B-Instruct
/home/quist/source/llm_train/venv/lib/python3.12/site-packages/transformers/models/auto/modeling_auto.py:2284: FutureWarning: The class `AutoModelForVision2Seq` is deprecated and will be removed in v5.0. Please use `AutoModelForImageTextToText` instead.
  warnings.warn(

MERGING LORA ADAPTERS INTO BASE MODEL

[1/3] Loading base model: Qwen/Qwen2.5-VL-7B-Instruct
/home/quist/source/llm_train/venv/lib/python3.12/site-packages/transformers/models/auto/modeling_auto.py:2284: FutureWarning: The class `AutoModelForVision2Seq` is deprecated and will be removed in v5.0. Please use `AutoModelForImageTextToText` instead.
  warnings.warn(
`torch_dtype` is deprecated! Use `dtype` instead!
`torch_dtype` is deprecated! Use `dtype` instead!
Loading checkpoint shards:  60%|██████████▊       | 3/5 [00:06<00:04,  2.33s/it]

: 

In [ ]:
# Step 3: Quantize to AWQ for efficient inference
print("\n" + "="*60)
print("STEP 3: QUANTIZING TO AWQ")
print("="*60)
! python quantize_to_awq.py \
    --model_dir checkpoints/qwen2_5_vl_lora_full/merged \
    --output_dir checkpoints/qwen2_5_vl_lora_full/awq_model \
    --num_calibration_samples 256

In [ ]:
# Step 3: Quantize to AWQ for efficient inference (DEV)
print("\n" + "="*60)
print("STEP 3: QUANTIZING TO AWQ (DEV)")
print("="*60)
! python quantize_to_awq.py \
    --model_dir checkpoints/qwen2_5_vl_lora_dev/merged \
    --output_dir checkpoints/qwen2_5_vl_lora_dev/awq_model \
    --num_calibration_samples 128

In [ ]:
# Environment check for quantization (versions + CUDA)
import sys
import torch
import transformers

print("Python:", sys.version)
print("Torch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("CUDA device:", torch.cuda.get_device_name(0))
    print("bf16 support:", torch.cuda.is_bf16_supported())
else:
    print("Note: Quantization may still run on CPU, but AWQ calibration is faster on GPU.")

## AWQ Quantization Environment Setup (Optional)
To avoid package conflicts with training, create a clean environment specifically for AWQ quantization.

Recommended steps:

1) Create and activate a new virtual environment
2) Install a compatible PyTorch (CPU or CUDA as needed)
3) Install Transformers and AWQ dependencies
4) Run the quantization script in this environment

In [ ]:
# Set up a clean AWQ env (Linux bash)
# Env name: awq-venv (top-level folder, not hidden)
# Note: llmcompressor has strict version requirements, so we use a clean venv

# 1) Create clean venv
! rm -rf awq-venv && python3 -m venv awq-venv

# 2) Install PyTorch 2.5.1 with CUDA 12.1 (compatible with CUDA 13.0)
! source awq-venv/bin/activate && pip install --upgrade pip
! source awq-venv/bin/activate && pip install torch==2.5.1 torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121

# 3) Install quantization dependencies (transformers first, then llmcompressor, then upgrade transformers)
! source awq-venv/bin/activate && pip install transformers==4.44.2 accelerate safetensors "llmcompressor<0.8"
! source awq-venv/bin/activate && pip install --upgrade transformers

# 4) Verify installation
! source awq-venv/bin/activate && python -c "import torch; print('✓ Torch', torch.__version__, 'CUDA:', torch.cuda.is_available()); import llmcompressor; print('✓ llmcompressor OK')"

In [ ]:
# Step 4: Verify the pipeline completed successfully
print("\n" + "="*60)
print("PIPELINE COMPLETE")
print("="*60)

import os
from pathlib import Path

checkpoint_dir = Path("checkpoints/qwen2_5_vl_lora_full")

# Check what was saved
print("\nGenerated artifacts:")
if (checkpoint_dir / "adapter_model").exists():
    adapter_size = sum(f.stat().st_size for f in (checkpoint_dir / "adapter_model").rglob("*") if f.is_file()) / (1024**2)
    print(f"✓ LoRA adapters: {adapter_size:.1f} MB")

if (checkpoint_dir / "merged").exists():
    merged_size = sum(f.stat().st_size for f in (checkpoint_dir / "merged").rglob("*") if f.is_file()) / (1024**2)
    print(f"✓ Merged model: {merged_size:.1f} MB")

if (checkpoint_dir / "awq_model").exists():
    awq_size = sum(f.stat().st_size for f in (checkpoint_dir / "awq_model").rglob("*") if f.is_file()) / (1024**2)
    print(f"✓ AWQ quantized model: {awq_size:.1f} MB")

print("\n📊 Next steps for production:")
print(f"1. Deploy with vLLM:")
print(f"   vllm serve {checkpoint_dir}/awq_model --quantization awq --gpu-memory-utilization 0.9")
print(f"\n2. Example inference:")
print(f"   from vllm import LLM")
print(f'   llm = LLM(model="{checkpoint_dir}/awq_model", quantization="awq")')
print(f'   output = llm.generate(["Extract book info from spine image..."])')
print(f"\n✨ Your fine-tuned model is ready for production!")

In [ ]:
# Verify DEV pipeline artifacts
print("\n" + "="*60)
print("VERIFY DEV PIPELINE")
print("="*60)

from pathlib import Path
checkpoint_dir = Path("checkpoints/qwen2_5_vl_lora_dev")

print("\nGenerated DEV artifacts:")
if (checkpoint_dir / "adapter_model").exists():
    adapter_size = sum(f.stat().st_size for f in (checkpoint_dir / "adapter_model").rglob("*") if f.is_file()) / (1024**2)
    print(f"✓ LoRA adapters (dev): {adapter_size:.1f} MB")

if (checkpoint_dir / "merged").exists():
    merged_size = sum(f.stat().st_size for f in (checkpoint_dir / "merged").rglob("*") if f.is_file()) / (1024**2)
    print(f"✓ Merged model (dev): {merged_size:.1f} MB")

if (checkpoint_dir / "awq_model").exists():
    awq_size = sum(f.stat().st_size for f in (checkpoint_dir / "awq_model").rglob("*") if f.is_file()) / (1024**2)
    print(f"✓ AWQ quantized model (dev): {awq_size:.1f} MB")

print("\nIf AWQ completed, you can serve via vLLM:")
print(f"   vllm serve {checkpoint_dir}/awq_model --quantization awq --gpu-memory-utilization 0.9")